# Add SpatialData

- Convert the Cohort of Fobs and labels to a single `SpatialData` object. 
- Save the `SpatialData` object to `LaminDB`.


## Notebook Preferences

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Importing Libraries

In [2]:
from upath import UPath
import buckaroo  # noqa: F401
import pandas as pd
import natsort as ns
import lamindb as ln
from nbl.util import DaskLocalCluster, reset_table_index
import nbl
import spatialdata as sd

Buckaroo has been enabled as the default DataFrame viewer.  To return to default dataframe visualization use `from buckaroo import disable; disable()`
💡 connected lamindb: srivarra/nbl_assets


In [3]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [4]:
ln.settings.transform.stem_uid = "sDPLFLgnLcbi"
ln.settings.transform.version = "1"
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
run = ln.track()
run.transform

💡 notebook imports: buckaroo==0.6.12 lamindb==0.75.1 natsort==8.4.0 nbl==0.1.dev2+gcbd4c32.d20240811 pandas==2.2.2 spatialdata==0.2.2.dev31+ga193a25 universal_pathlib==0.2.2
💡 loaded: Transform(uid='sDPLFLgnLcbi5zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-08-12 17:03:20 UTC')
💡 loaded: Run(uid='QOSQX4xGWV6rtg4SDd2y', started_at='2024-08-12 17:05:27 UTC', is_consecutive=True, transform_id=2, created_by_id=1)


Transform(uid='sDPLFLgnLcbi5zKv', version='1', name='Add SpatialData', key='01 - Add SpatialData', type='notebook', created_by_id=1, updated_at='2024-08-12 17:03:20 UTC')

In [5]:
ln.setup.settings.instance

Current instance: srivarra/nbl_assets
- owner: srivarra
- name: nbl_assets
- storage root: /Users/srivarra/Davis Lab/neuroblastoma/nbl/data/db
- storage region: None
- db: postgresql://srivarra:***@***:5555/nbl_db
- schema: {'bionty'}
- git_repo: None

In [6]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 10,Total memory: 64.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50963,Workers: 10
Dashboard: http://127.0.0.1:8787/status,Total threads: 10
Started: Just now,Total memory: 64.00 GiB
Comm: tcp://127.0.0.1:50991,Total threads: 1
Dashboard: http://127.0.0.1:50997/status,Memory: 6.40 GiB
Nanny: tcp://127.0.0.1:50966,


## Convert FOVs to SpatialData

### Set up Paths

In [7]:
original_data_path = UPath("../../../data/raw/original_data/nbl_cohort")
fov_dir: UPath = original_data_path / "images"
label_dir: UPath = original_data_path / "segmentation" / "labels"

In [8]:
hu_data_path: UPath = ln.settings.storage.root / "Hu.zarr"
nbl_data_path: UPath = ln.settings.storage.root / "nbl.zarr"

### Convert Control Cohort to SpatialData - Hu Data

#### Convert Cohort

In [9]:
nbl.io.convert_cohort(fov_dir=fov_dir, label_dir=label_dir, filter_fovs=r"Hu-*", file_path=hu_data_path)

💡 connected lamindb: srivarra/nbl_assets
💡 connected lamindb: srivarra/nbl_assets
💡 connected lamindb: srivarra/nbl_assets
💡 connected lamindb: srivarra/nbl_assets
💡 connected lamindb: srivarra/nbl_assets
INFO     The Zarr backing store has been changed from None the new file path: /Users/srivarra/Davis                
         Lab/neuroblastoma/nbl/data/db/Hu.zarr                                                                     


In [10]:
hu_sdata = sd.read_zarr(store=hu_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [11]:
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell")
nbl.tl.aggregate_images_by_labels(sdata=hu_sdata, label_type="nuclear", table_name="nuclear")

2024-08-12 10:06:02,166 - distributed.scheduler - WARNING - Detected different `run_spec` for key 'original-from-zarr-5f6ad2307a1665f95868e512e96a7aa8' between two consecutive calls to `update_graph`. This can cause failures and deadlocks down the line. Please ensure unique key names. If you are using a standard dask collections, consider releasing all the data before resubmitting another computation. More details and help can be found at https://github.com/dask/dask/issues/9888. 
Debugging information
---------------------
old task state: released
old run_spec: (<function execute_task at 0x33a8ad9e0>, (<zarr.core.Array '/0' (1024, 1024) int64 read-only>,), {})
new run_spec: (<function execute_task at 0x33a8ad9e0>, (<zarr.core.Array '/0' (1024, 1024) int64 read-only>,), {})
old token: ('tuple', [('80935a1067ef908b', []), ('tuple', [('c7285342004d89c8', ['53d3d5d7c3af4670'])]), ('dict', [])])
new token: ('tuple', [('80935a1067ef908b', []), ('tuple', [('c7285342004d89c8', ['34c96acdcadb1

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWar

💡 connected lamindb: srivarra/nbl_assets
💡 connected lamindb: srivarra/nbl_assets


/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


💡 connected lamindb: srivarra/nbl_assets


/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWar

💡 connected lamindb: srivarra/nbl_assets


/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


💡 connected lamindb: srivarra/nbl_assets


In [12]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [13]:
nbl.tl.regionprops(sdata=hu_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)
nbl.tl.regionprops(sdata=hu_sdata, label_type="nuclear", table_name="nuclear", properties=properties)

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/_core/_elements.py:116: UserWarning: Key `whole_cell` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/_core/_elements.py:116: UserWarning: Key `nuclear` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


### Convert Sample Cohort to SpatialData - NBL Data

#### Convert Cohort

In [14]:
nbl.io.convert_cohort(fov_dir=fov_dir, filter_fovs=r"NBL-\d+-R\d+C\d+", label_dir=label_dir, file_path=nbl_data_path)

INFO     The Zarr backing store has been changed from None the new file path: /Users/srivarra/Davis                
         Lab/neuroblastoma/nbl/data/db/nbl.zarr                                                                    


In [15]:
nbl_sdata = sd.read_zarr(store=nbl_data_path)

#### Aggregate Images by Labels and Compute Region Properties

In [16]:
nbl.tl.aggregate_images_by_labels(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell")

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/models/models.py:996: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:430: FutureWar

In [17]:
properties = [
    "label",
    "centroid",
    "area",
    "perimeter",
    "axis_major_length",
    "axis_minor_length",
    "eccentricity",
    "orientation",
]

In [18]:
nbl.tl.regionprops(sdata=nbl_sdata, label_type="whole_cell", table_name="whole_cell", properties=properties)

/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 67.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/Users/srivarra/Davis Lab/neuroblastoma/nbl/.venv/lib/python3.11/site-packages/spatialdata/_core/_elements.py:116: UserWarning: Key `whole_cell` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)


## Add Pixie Clusters to NBL SpatialData

In [19]:
pixie_clusters_path = (
    original_data_path / "segmentation" / "cell_table" / "cell_table_size_normalized_cell_labels_noCD117.csv"
)

In [20]:
pixie_clusters_df = pd.read_csv(pixie_clusters_path).astype({"label": int, "cell_meta_cluster": "category"})

In [21]:
all_fovs = ns.natsorted(nbl_sdata.coordinate_systems)

In [22]:
def get_pixie_clusters(df, fovs: str, suffix="whole_cell") -> pd.DataFrame:
    """Gets pixie clusters from the two clustering csv files.

    Parameters
    ----------
    df : pd.DataFrame
        The Pixie cluster DataFrame
    fov : str
        The FOV to subset on

    Returns
    -------
    pd.DataFrame
        A dataframe containing the two clusters and a column for the segmentation label.
    """
    out_df = []
    for fov in fovs:
        fov_rncm = fov.split("-")[-1].split("_")[0]
        no_cd117_pixie: pd.DataFrame = df[df["fov"].str.split("_").map(lambda x: x[-1]) == fov_rncm]
        result_df = (
            no_cd117_pixie.assign(region=f"{fov}_{suffix}", fov=fov)
            .rename(columns={"label": "instance_id", "cell_meta_cluster": "pixie_cluster"})
            .astype(dtype={"instance_id": int, "region": "category", "pixie_cluster": "category"})
            .filter(items=["instance_id", "region", "fov", "pixie_cluster"])
        )
        out_df.append(result_df)

    return pd.concat(out_df)

In [23]:
pixie_df = pixie_clusters_df.pipe(func=get_pixie_clusters, fovs=all_fovs, suffix="whole_cell")

In [24]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"]
    .obs.merge(
        right=pixie_df,
        right_on=["instance_id", "region"],
        left_on=["instance_id", "region"],
    )
    .pipe(reset_table_index)
)

In [25]:
nbl_sdata.tables["whole_cell"].obs

BuckarooWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative'], 'post_pr…

## Add Clinical Information

### Load Clinical Data from LaminDB

In [26]:
clinical_data: pd.DataFrame = ln.Artifact.filter(key__contains="clinical_data").one().load()

In [27]:
cols_to_drop = ["Clinical presentation", "treatment btw biopsies"]

In [28]:
filtered_clincial_data = clinical_data.drop(columns=cols_to_drop)

In [29]:
nbl_sdata.tables["whole_cell"].obs = (
    nbl_sdata.tables["whole_cell"].obs.merge(right=filtered_clincial_data, on="fov").pipe(reset_table_index)
)
nbl_sdata.tables["whole_cell"].strings_to_categoricals()

### `Arcsinh` Transform the NBL Whole Cell Table

In [30]:
nbl.pp.arcsinh_transform(
    sdata=nbl_sdata, table_names="whole_cell", shift_factor=0, scale_factor=150, replace_X=True, write=True
)

In [31]:
hu_artifact = ln.Artifact(data=hu_data_path, type="dataset", key="Hu.zarr", description="Control Tissue")

hu_artifact.save(upload=True)

Artifact(uid='HbqHTeg2N2ACNvG9LIbO', description='Control Tissue', key='Hu.zarr', suffix='.zarr', type='dataset', size=434280394, hash='kjVDALg9vI0XU1rkLzuAyw', _hash_type='md5-d', n_objects=466, visibility=1, _key_is_virtual=False, created_by_id=1, storage_id=1, transform_id=2, run_id=2, updated_at='2024-08-12 17:15:28 UTC')

In [32]:
nbl_artifact = ln.Artifact(data=nbl_data_path, type="dataset", key="nbl.zarr", description="NBL Tissue Samples")

nbl_artifact.save(upload=True)

Artifact(uid='UPR9xMFgQw4sByO6pd9Q', description='NBL Tissue Samples', key='nbl.zarr', suffix='.zarr', type='dataset', size=12717502873, hash='poatLM8BOW29bolDuUx4hg', _hash_type='md5-d', n_objects=6638, visibility=1, _key_is_virtual=False, created_by_id=1, storage_id=1, transform_id=2, run_id=2, updated_at='2024-08-12 17:15:38 UTC')

In [34]:
ln.finish()

✅ cell execution numbers increase consecutively
